# Chạy thử AI gợi ý món — chỉ bấm ▶ một lần

**Bản dễ dùng v2 — ô chạy độc lập, không cần chạy các ô khai báo hàm trước.**

1. Trong ô bên dưới, giữ `DATA_MODE = demo` để chạy thử.
2. Nhập món đang chọn ở `MON_DANG_CHON`, ngăn cách bằng dấu phẩy. Ví dụ: **Cơm trắng, Rau xào**.
3. Bấm **▶** ở bên trái **chính ô bên dưới**. Chờ đến khi hiện **4/4 — ĐÃ CHẠY XONG**.
4. Đọc **ĐANG CHỌN** và **GỢI Ý** ngay dưới ô. Muốn thử món khác, sửa tên rồi bấm lại ▶.

Lần đầu cần cài thư viện nên có thể mất vài phút. Bạn sẽ thấy tiến độ 1/4 → 4/4.
Để trống tên món sẽ lấy một ví dụ từ các luật vừa học. Có thể dùng tên món hoặc mã món.

**Demo dùng 2.000 giỏ mô phỏng**, không phải dữ liệu kinh doanh thực tế.
Chọn `csv` để tải file xuất từ ứng dụng; file chỉ cần dữ liệu giỏ món, không cần tên/SĐT khách.


In [ ]:
# @title ▶ BẤM NÚT NÀY ĐỂ CHẠY AI { display-mode: "form" }
DATA_MODE = "demo" # @param ["demo", "csv"]
MON_DANG_CHON = "Cơm trắng, Rau xào" # @param {type:"string"}
DEMO_BASKETS = 2000 # @param {type:"integer"}

"""Standard-library-only controller for the single runnable Colab cell."""

from datetime import datetime, timezone
from html import escape
from importlib import metadata
import json
from pathlib import Path
import shutil
import subprocess
import sys
from tempfile import TemporaryDirectory
import warnings


def ensure_training_packages(requirements):
    missing = []
    for package, version in requirements.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            missing.append(f"{package}=={version}")
    if missing:
        print("Đang cài thư viện. Bước này có thể mất vài phút ở lần đầu...", flush=True)
        process = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                                 capture_output=True, text=True, encoding="utf-8", errors="replace")
        if process.returncode:
            raise RuntimeError("Chưa cài được thư viện. Chi tiết lỗi đầu tiên:\n" + process.stdout + process.stderr)


def execute_notebook(*, mode, cart_text, baskets, source, requirements, csv_path=None, work_dir=None, download=True):
    if mode not in {"demo", "csv"}:
        raise ValueError("DATA_MODE chỉ nhận demo hoặc csv.")
    if not isinstance(baskets, int) or baskets < 2:
        raise ValueError("Số giỏ mô phỏng phải là số nguyên từ 2 trở lên.")
    # Colab's Jupyter transport may emit this Python 3.13 warning for every print.
    # Leave installation/training exceptions and all other warnings visible.
    warnings.filterwarnings("ignore", message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
                            category=DeprecationWarning, module=r"jupyter_client\.session")
    try:
        from google.colab import files
    except ImportError:
        files = None
    print("1/4 — Chuẩn bị thư viện", flush=True)
    ensure_training_packages(requirements)
    root = Path(work_dir) if work_dir else Path("/content/qlnh_apriori" if files else "artifacts/ai/notebook")
    root.mkdir(parents=True, exist_ok=True)
    output = root / ("apriori_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    print("2/4 — Chuẩn bị dữ liệu", flush=True)
    if mode == "csv":
        if csv_path is None and files:
            print("Chọn file CSV đã xuất từ ứng dụng.", flush=True)
            uploaded = files.upload()
            candidates = [name for name in uploaded if name.lower().endswith(".csv")]
            if len(candidates) != 1:
                raise ValueError("Chọn đúng một file CSV giao dịch.")
            csv_path = candidates[0]
        if csv_path is None or not Path(csv_path).is_file():
            raise ValueError("Chưa có file CSV. Chọn demo để chạy thử hoặc tải CSV lên.")
    job = {"mode": mode, "cart_text": cart_text, "baskets": baskets, "output": str(output.resolve()),
           "csv_path": str(Path(csv_path).resolve()) if csv_path else None}
    print("3/4 — Chạy Apriori và kiểm tra gợi ý", flush=True)
    with TemporaryDirectory(prefix="qlnh-apriori-worker-") as directory:
        worker, job_file = Path(directory) / "worker.py", Path(directory) / "job.json"
        worker.write_text(source, encoding="utf-8")
        job_file.write_text(json.dumps(job, ensure_ascii=False), encoding="utf-8")
        process = subprocess.run([sys.executable, "-u", str(worker), str(job_file)], capture_output=True,
                                 text=True, encoding="utf-8", errors="replace")
        if process.returncode:
            raise RuntimeError("Bước chạy Apriori chưa hoàn tất. Chi tiết lỗi:\n" + process.stdout + process.stderr)
    model = json.loads((output / "apriori_model.json").read_text(encoding="utf-8"))
    inference = json.loads((output / "suggestions.json").read_text(encoding="utf-8"))
    names = {item["dish_id"]: item["dish_name"] for item in model["catalog"]}
    selected = [names[item] for item in inference["cart"]]
    suggestions = [names[item] for item in inference["suggestions"]]
    print("4/4 — ĐÃ CHẠY XONG", flush=True)
    print("Nguồn dữ liệu:", model["data_source"], "| Số giỏ:", inference["basket_count"])
    print("Số luật tìm được:", model["training"]["rule_count"])
    print("ĐANG CHỌN:", ", ".join(selected) or "Chưa chọn món")
    print("GỢI Ý:", ", ".join(suggestions) or "Chưa có gợi ý phù hợp")
    if inference["unknown"]:
        print("Không tìm thấy món:", ", ".join(inference["unknown"]), "— xem danh mục bên dưới và sửa MON_DANG_CHON.")
    if inference["ambiguous"]:
        print("Tên trùng nhiều món:", ", ".join(inference["ambiguous"]), "— nhập mã món hoặc dish_id để chọn rõ.")
    if mode == "demo":
        print("Đây là dữ liệu mô phỏng; kết quả này chỉ dùng chạy thử quy trình.")
    print("Đánh giá:", json.dumps(model["evaluation"], ensure_ascii=False, indent=2))
    try:
        from IPython.display import HTML, display
    except ImportError:
        for item in model["catalog"]:
            print(item["dish_id"], "—", item["dish_name"])
    else:
        rows = "".join("<tr><td>" + escape(item["dish_id"]) + "</td><td>" + escape(item["dish_code"])
                       + "</td><td>" + escape(item["dish_name"]) + "</td></tr>" for item in model["catalog"])
        display(HTML("<p><b>Đang chọn:</b> " + escape(", ".join(selected)) + "</p><p><b>Gợi ý:</b> "
                     + escape(", ".join(suggestions) or "Chưa có gợi ý phù hợp")
                     + "</p><details><summary>Danh mục món để thử tiếp</summary><table><tr><th>dish_id</th><th>Mã</th><th>Tên món</th></tr>"
                     + rows + "</table></details>"))
    archive = shutil.make_archive(str(output), "zip", root_dir=output)
    print("Kết quả:", output.resolve())
    if files and download:
        files.download(archive)
    return model, output, inference, archive


TRAINING_SOURCE = 'SCHEMA_VERSION = 1\nTRANSACTION_FIELDS = (\n    "schema_version", "data_source", "basket_id", "basket_closed_at",\n    "dish_id", "dish_code", "dish_name", "quantity",\n)\nDATA_SOURCES = {"application_export", "synthetic_demo"}\n\n\n"""Validated Apriori inference using only the Python standard library."""\n\nimport json\nimport math\nfrom pathlib import Path\n\n\nclass InvalidAprioriModel(ValueError):\n    pass\n\n\ndef validate_model(payload):\n    if not isinstance(payload, dict) or type(payload.get("schema_version")) is not int or payload["schema_version"] != 1:\n        raise InvalidAprioriModel("Unsupported model schema")\n    if payload.get("algorithm") != "Apriori" or payload.get("data_source") not in {"application_export", "synthetic_demo"}:\n        raise InvalidAprioriModel("Unsupported model algorithm or source")\n    catalog, rules = payload.get("catalog"), payload.get("rules")\n    if not isinstance(catalog, list) or not 1 <= len(catalog) <= 5000 or not isinstance(rules, list) or len(rules) > 5000:\n        raise InvalidAprioriModel("Invalid model size")\n    identifiers, codes = set(), set()\n    for item in catalog:\n        if not isinstance(item, dict) or any(not isinstance(item.get(key), str) or not item[key].strip() or len(item[key]) > 200\n                                             for key in ("dish_id", "dish_code", "dish_name")):\n            raise InvalidAprioriModel("Invalid catalog item")\n        if item["dish_id"] in identifiers or item["dish_code"] in codes:\n            raise InvalidAprioriModel("Duplicate catalog identifier")\n        identifiers.add(item["dish_id"])\n        codes.add(item["dish_code"])\n    normalized = []\n    for rule in rules:\n        if not isinstance(rule, dict):\n            raise InvalidAprioriModel("Invalid rule")\n        left, right = rule.get("antecedents"), rule.get("consequents")\n        if any(not isinstance(values, list) or not 1 <= len(values) <= 10 or\n               any(not isinstance(value, str) or value not in identifiers for value in values)\n               for values in (left, right)):\n            raise InvalidAprioriModel("Invalid rule items")\n        if len(set(left)) != len(left) or len(set(right)) != len(right) or set(left) & set(right):\n            raise InvalidAprioriModel("Invalid rule overlap")\n        metrics = {}\n        for key in ("support", "confidence", "lift"):\n            value = rule.get(key)\n            if type(value) not in (int, float) or not math.isfinite(value) or value < 0 or (key != "lift" and value > 1):\n                raise InvalidAprioriModel("Invalid rule metrics")\n            metrics[key] = float(value)\n        normalized.append({"antecedents": sorted(left), "consequents": sorted(right), **metrics})\n    normalized.sort(key=lambda rule: (-rule["confidence"], -rule["lift"], -rule["support"], rule["antecedents"], rule["consequents"]))\n    return {**payload, "rules": normalized}\n\n\ndef load_model(path):\n    path = Path(path)\n    if path.stat().st_size > 2_000_000:\n        raise InvalidAprioriModel("Model exceeds size limit")\n    try:\n        payload = json.loads(path.read_text(encoding="utf-8-sig"))\n    except (ValueError, UnicodeError, RecursionError) as error:\n        raise InvalidAprioriModel("Invalid model JSON") from error\n    return validate_model(payload)\n\n\ndef recommend(cart, rules, *, k=3, allowed_items=None):\n    if k < 1:\n        raise ValueError("k phải dương.")\n    selected, suggestions = set(cart), []\n    allowed = set(allowed_items) if allowed_items is not None else None\n    for rule in rules:\n        if set(rule["antecedents"]) <= selected:\n            for item in rule["consequents"]:\n                if item not in selected and item not in suggestions and (allowed is None or item in allowed):\n                    suggestions.append(item)\n                    if len(suggestions) == k:\n                        return suggestions\n    return suggestions\n\n\n"""Reproducible fictional baskets; never connects to Django or its database."""\n\nimport csv\nfrom datetime import date, datetime, time, timedelta, timezone\nfrom pathlib import Path\nimport random\n\n\n\nDEMO_DISHES = (\n    ("DEMO01", "Cơm gà"), ("DEMO02", "Canh rau"), ("DEMO03", "Trà đá"),\n    ("DEMO04", "Bún bò"), ("DEMO05", "Nem rán"), ("DEMO06", "Nước cam"),\n    ("DEMO07", "Cá kho"), ("DEMO08", "Cơm trắng"), ("DEMO09", "Rau xào"),\n    ("DEMO10", "Lẩu nấm"), ("DEMO11", "Mì"), ("DEMO12", "Chè"),\n)\n\n\ndef demo_rows(*, baskets=2000, days=180, seed=42, start=date(2026, 1, 1)):\n    if baskets < 2 or days < 2:\n        raise ValueError("Cần ít nhất 2 giỏ món và 2 ngày để chia tập theo thời gian.")\n    rng = random.Random(seed)\n    names = dict(DEMO_DISHES)\n    bundles = [codes for codes in (\n        ("DEMO01", "DEMO02", "DEMO03"), ("DEMO04", "DEMO05", "DEMO06"),\n        ("DEMO07", "DEMO08", "DEMO09"), ("DEMO10", "DEMO11", "DEMO03"),\n    )]\n    for index in range(baskets):\n        day = start + timedelta(days=index * days // baskets)\n        closed_at = datetime.combine(day, time(rng.choice((11, 12, 18, 19, 20)), rng.randrange(60)),\n                                     tzinfo=timezone(timedelta(hours=7)))\n        # These associations are deliberately planted for a reproducible demo.\n        bundle = rng.choice(bundles)\n        chosen = {bundle[0]}\n        chosen.update(code for code in bundle[1:] if rng.random() < 0.8)\n        if rng.random() < 0.3:\n            chosen.add(rng.choice(DEMO_DISHES)[0])\n        if rng.random() < 0.2:\n            chosen.add("DEMO12")\n        for code in sorted(chosen):\n            yield {\n                "schema_version": SCHEMA_VERSION, "data_source": "synthetic_demo",\n                "basket_id": f"DEMO-{index + 1:06d}", "basket_closed_at": closed_at.isoformat(),\n                "dish_id": code, "dish_code": code, "dish_name": names[code],\n                "quantity": rng.choice((1, 1, 1, 2, 3)),\n            }\n\n\ndef write_demo_csv(path, **options):\n    rows = iter(demo_rows(**options))\n    first = next(rows)  # Validate options before creating any file.\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("x", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=TRANSACTION_FIELDS)\n        writer.writeheader()\n        writer.writerow(first)\n        writer.writerows(rows)\n    return path\n\n\n"""Apriori experiment shared by the CLI and the self-contained Colab notebook."""\n\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport mlxtend\nfrom mlxtend.frequent_patterns import apriori, association_rules\nfrom mlxtend.preprocessing import TransactionEncoder\n\n\n\ndef load_transactions(path):\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")\n    missing = set(TRANSACTION_FIELDS) - set(frame.columns)\n    if missing:\n        raise ValueError(f"CSV thiếu các cột: {\', \'.join(sorted(missing))}")\n    frame = frame[list(TRANSACTION_FIELDS)].copy()\n    if frame.empty:\n        raise ValueError("CSV chưa có giỏ món hợp lệ. Xuất thêm dữ liệu hoặc chạy chế độ demo.")\n    for field in TRANSACTION_FIELDS:\n        frame[field] = frame[field].str.strip()\n        if frame[field].eq("").any():\n            raise ValueError(f"Cột {field} có giá trị trống.")\n    if set(frame["schema_version"]) != {str(SCHEMA_VERSION)}:\n        raise ValueError("Phiên bản schema CSV chưa được hỗ trợ.")\n    sources = set(frame["data_source"])\n    if len(sources) != 1 or not sources <= DATA_SOURCES:\n        raise ValueError("Chỉ dùng một nguồn application_export hoặc synthetic_demo trong mỗi lần chạy.")\n    if not frame["basket_closed_at"].str.contains(r"(?:Z|[+-]\\d{2}:\\d{2})$", regex=True).all():\n        raise ValueError("basket_closed_at phải có múi giờ, ví dụ 2026-01-01T12:00:00+07:00.")\n    frame["basket_closed_at"] = pd.to_datetime(frame["basket_closed_at"], format="ISO8601", utc=True)\n    quantity = pd.to_numeric(frame["quantity"], errors="raise")\n    if not (np.isfinite(quantity) & quantity.gt(0) & quantity.mod(1).eq(0)).all():\n        raise ValueError("quantity phải là số nguyên dương hữu hạn.")\n    frame["quantity"] = quantity\n    if frame.groupby("basket_id")["basket_closed_at"].nunique().gt(1).any():\n        raise ValueError("Một basket_id phải có cùng thời điểm đóng giỏ ở mọi dòng.")\n    for field in ("dish_code", "dish_name"):\n        if frame.groupby("dish_id")[field].nunique().gt(1).any():\n            raise ValueError(f"Một dish_id có nhiều giá trị {field}; cần chuẩn hóa lại danh mục.")\n    return frame.drop_duplicates().reset_index(drop=True)\n\n\ndef split_baskets(frame, *, test_fraction=0.2):\n    if not 0 < test_fraction < 1:\n        raise ValueError("test_fraction phải nằm giữa 0 và 1.")\n    baskets = frame.groupby("basket_id", sort=True).agg(\n        closed_at=("basket_closed_at", "first"),\n        items=("dish_id", lambda values: tuple(sorted(set(values)))),\n    ).reset_index().sort_values(["closed_at", "basket_id"])\n    baskets["business_day"] = baskets["closed_at"].dt.tz_convert("Asia/Ho_Chi_Minh").dt.date\n    days = sorted(baskets["business_day"].unique())\n    if len(days) < 2:\n        raise ValueError("Cần ít nhất 2 ngày có giỏ món để tách tập kiểm tra theo thời gian.")\n    test_days = min(len(days) - 1, max(1, math.ceil(len(days) * test_fraction)))\n    cutoff = days[-test_days]\n    train = baskets[baskets["business_day"] < cutoff].copy()\n    test = baskets[baskets["business_day"] >= cutoff].copy()\n    return train, test, cutoff\n\n\ndef fit_rules(baskets, *, min_support=0.04, min_confidence=0.3, min_lift=1.05, max_len=3, max_rules=500):\n    if not (0 < min_support <= 1 and 0 < min_confidence <= 1 and math.isfinite(min_lift) and min_lift > 0):\n        raise ValueError("Ngưỡng support/confidence phải thuộc (0, 1], lift phải dương hữu hạn.")\n    if max_len < 2 or max_rules < 1:\n        raise ValueError("max_len phải từ 2 và max_rules phải dương.")\n    transactions = baskets["items"].tolist()\n    if not transactions:\n        raise ValueError("Tập huấn luyện trống.")\n    encoder = TransactionEncoder()\n    matrix = encoder.fit(transactions).transform(transactions)\n    encoded = pd.DataFrame(matrix, columns=encoder.columns_, dtype=bool)\n    frequent = apriori(encoded, min_support=min_support, use_colnames=True, max_len=max_len)\n    if frequent.empty or not frequent["itemsets"].map(len).ge(2).any():\n        return [], frequent\n    discovered = association_rules(frequent, num_itemsets=len(encoded), metric="confidence",\n                                   min_threshold=min_confidence, return_metrics=["support", "confidence", "lift"])\n    discovered = discovered[discovered["consequents"].map(len).eq(1) & discovered["lift"].ge(min_lift)]\n    rules = [{\n        "antecedents": sorted(row.antecedents), "consequents": sorted(row.consequents),\n        "support": float(row.support), "confidence": float(row.confidence), "lift": float(row.lift),\n    } for row in discovered.itertuples()]\n    rules.sort(key=lambda rule: (-rule["confidence"], -rule["lift"], -rule["support"], rule["antecedents"], rule["consequents"]))\n    return rules[:max_rules], frequent\n\n\ndef evaluate_rules(train, test, rules, *, k=3):\n    popularity = Counter(item for items in train["items"] for item in set(items))\n    ranked_popular = sorted(popularity, key=lambda item: (-popularity[item], item))\n    eligible = hits = baseline_hits = covered = 0\n    for basket in test.itertuples():\n        if len(basket.items) < 2:\n            continue\n        hidden_index = int.from_bytes(hashlib.sha256(basket.basket_id.encode()).digest()[:8], "big") % len(basket.items)\n        hidden = basket.items[hidden_index]\n        observed = set(basket.items) - {hidden}\n        predictions = recommend(observed, rules, k=k)\n        baseline = [item for item in ranked_popular if item not in observed][:k]\n        eligible += 1\n        covered += bool(predictions)\n        hits += hidden in predictions\n        baseline_hits += hidden in baseline\n    return {\n        "protocol": "chronological_business_days_then_hide_one_item_sha256_basket_id",\n        "k": k, "test_baskets": len(test), "eligible_baskets": eligible,\n        "single_item_baskets_skipped": len(test) - eligible,\n        "hit_rate_at_k": hits / eligible if eligible else None,\n        "recommendation_coverage": covered / eligible if eligible else None,\n        "popularity_baseline_hit_rate_at_k": baseline_hits / eligible if eligible else None,\n        "note": "Giấu một món trong giỏ hoàn tất để kiểm tra; không phải đánh giá thứ tự chọn món thực tế.",\n    }\n\n\ndef run_experiment(csv_path, output_dir, *, min_support=0.04, min_confidence=0.3, min_lift=1.05,\n                   max_len=3, max_rules=500, test_fraction=0.2, k=3):\n    frame = load_transactions(csv_path)\n    train, test, cutoff = split_baskets(frame, test_fraction=test_fraction)\n    parameters = {"min_support": min_support, "min_confidence": min_confidence,\n                  "min_lift": min_lift, "max_len": max_len, "max_rules": max_rules}\n    rules, frequent = fit_rules(train, **parameters)\n    metrics = evaluate_rules(train, test, rules, k=k)\n    source = frame["data_source"].iloc[0]\n    catalog = frame[["dish_id", "dish_code", "dish_name"]].drop_duplicates("dish_id").sort_values("dish_id")\n    model = {\n        "schema_version": SCHEMA_VERSION, "algorithm": "Apriori", "data_source": source,\n        "created_at": datetime.now(timezone.utc).isoformat(),\n        "versions": {"mlxtend": mlxtend.__version__, "pandas": pd.__version__, "numpy": np.__version__},\n        "input_sha256": hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),\n        "parameters": {**parameters, "test_fraction": test_fraction, "k": k},\n        "training": {"baskets": len(train), "holdout_baskets": len(test),\n                     "holdout_start_business_day": cutoff.isoformat(), "rule_count": len(rules),\n                     "train_start": train["closed_at"].min().isoformat(), "train_end": train["closed_at"].max().isoformat()},\n        "limitations": [\n            "Dữ liệu synthetic_demo có các cặp món được tạo chủ ý, chỉ dùng kiểm tra quy trình.",\n            "application_export xác nhận nguồn ứng dụng, không xác nhận đây là dữ liệu kinh doanh thực.",\n            "Support/confidence/lift đo liên hệ trong tập train, không phải độ chính xác ngoài thực tế.",\n            "Chọn ngưỡng trước khi xem holdout; thử nhiều ngưỡng trên holdout làm kết quả mang tính thăm dò.",\n            "Khi tích hợp web phải lọc lại món còn bán bằng trạng thái hiện tại của hệ thống.",\n        ],\n        "catalog": catalog.to_dict("records"), "evaluation": metrics, "rules": rules,\n    }\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=False)\n    for name, payload in (("apriori_model.json", model), ("evaluation.json", metrics)):\n        (output / name).write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    rule_frame = pd.DataFrame(rules, columns=["antecedents", "consequents", "support", "confidence", "lift"])\n    for column in ("antecedents", "consequents"):\n        rule_frame[column] = rule_frame[column].map(lambda items: json.dumps(items, ensure_ascii=False))\n    rule_frame.to_csv(output / "association_rules.csv", index=False, encoding="utf-8-sig")\n    frequent = frequent.copy()\n    frequent["itemsets"] = frequent["itemsets"].map(lambda items: json.dumps(sorted(items), ensure_ascii=False))\n    frequent.to_csv(output / "frequent_itemsets.csv", index=False, encoding="utf-8-sig")\n    for name, fold in (("train_baskets.csv", train), ("holdout_baskets.csv", test)):\n        fold = fold.copy()\n        fold["items"] = fold["items"].map(lambda items: json.dumps(items, ensure_ascii=False))\n        fold.to_csv(output / name, index=False, encoding="utf-8-sig")\n    return model\n\n\n"""Worker embedded in Colab so each experiment imports packages in a fresh process."""\n\nimport json\nfrom pathlib import Path\nimport sys\n\n\n\ndef notebook_job(job):\n    output = Path(job["output"])\n    if job["mode"] == "demo":\n        csv_path = write_demo_csv(output.parent / f"demo_{output.name}.csv",\n                                  baskets=job["baskets"], days=180, seed=42)\n    else:\n        csv_path = Path(job["csv_path"])\n    model = run_experiment(csv_path, output)\n    frame = load_transactions(csv_path)\n    catalog = model["catalog"]\n    lookup = {}\n    for item in catalog:\n        for value in (item["dish_id"], item["dish_code"], item["dish_name"]):\n            lookup.setdefault(value.casefold(), set()).add(item["dish_id"])\n    tokens = [value.strip() for value in job["cart_text"].split(",") if value.strip()]\n    cart, unknown, ambiguous = [], [], []\n    for token in tokens:\n        matches = lookup.get(token.casefold(), set())\n        if not matches:\n            unknown.append(token)\n        elif len(matches) > 1:\n            ambiguous.append(token)\n        else:\n            cart.extend(matches)\n    if not tokens and model["rules"]:\n        cart = list(model["rules"][0]["antecedents"])\n    cart = sorted(set(cart))\n    suggestions = recommend(cart, model["rules"]) if not unknown and not ambiguous else []\n    result = {"cart": cart, "suggestions": suggestions, "unknown": unknown, "ambiguous": ambiguous,\n              "basket_count": int(frame["basket_id"].nunique())}\n    (output / "suggestions.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    return result\n\n\nif __name__ == "__main__":\n    notebook_job(json.loads(Path(sys.argv[1]).read_text(encoding="utf-8")))\n'
TRAINING_REQUIREMENTS = {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'mlxtend': '0.23.4'}

MODEL, OUTPUT_DIR, INFERENCE, ARCHIVE = execute_notebook(
    mode=DATA_MODE, cart_text=MON_DANG_CHON, baskets=DEMO_BASKETS,
    source=TRAINING_SOURCE, requirements=TRAINING_REQUIREMENTS,
)


## Kết quả bạn nhận được

- **ĐANG CHỌN / GỢI Ý**: các món đầu vào và món thuật toán đề xuất.
- **Danh mục món để thử tiếp**: mở bảng để xem tên và mã món hợp lệ.
- File ZIP: chứa luật JSON, báo cáo đánh giá, các bảng CSV và `suggestions.json`.

Nếu không có luật phù hợp, chương trình ghi rõ chưa có gợi ý. Nếu nhập tên không tồn tại,
sửa tên theo danh mục rồi bấm lại ▶. Các chỉ số trên dữ liệu demo chỉ chứng minh quy trình.

Mã xử lý vẫn có trong ô chạy (có thể mở phần code để xem). Bản giải thích từng hàm nằm ở
`notebooks/02_apriori_giai_thich_code.ipynb`. Gợi ý trên giao diện nhà hàng là bước tiếp theo.
